# Multi-Agent Experiment (M0–M8)

Runs **multi-agent and ablation configurations** on the same stratified CUAD
sample used in `03_baseline_calibration.ipynb`. Supports running **multiple
models and experiment types** in a single execution, then performs statistical
comparison against baselines.

**Configurations:**
- `M0`: Classify-then-extract (two-stage: classifier + extractor)
- `M1`: Full multi-agent (orchestrator + 3 specialists + validation via LangGraph)
- `M6`: Combined specialist prompts in a single agent (architecture ablation)
- `M7`: Extract-then-verify (B4 extracts → specialist verifies/refines/recovers)
- `M8`: Ensemble (conservative B4 + aggressive specialist → resolver adjudicates)
- `M2`–`M5`: Reserved for future ablation studies

**Key hypotheses:**

| ID | Hypothesis | Test |
|----|-----------|------|
| H1 | Multi-agent beats single-agent baselines | F2(Mx) > F2(B4), McNemar p < 0.05 |
| H2 | Specialists help rare categories most | dF2_rare > dF2_common |
| H3 | Architecture matters, not just prompts | Mx > M6 significantly |
| H4 | Multi-agent produces auditable reasoning | Trace completeness > 90% |

For baselines (B1/B4), see `03_baseline_calibration.ipynb`.

In [5]:
# ============================================================
# CONFIGURATION — Edit these lists to run one or many experiments
# ============================================================

# Models to evaluate (cross-product with EXPERIMENT_TYPES)
MODELS = [
    #"gpt-4.1-nano",
    #"gpt-4.1-mini",
    #"gpt-4.1",
    #"o4-mini",
    #"o3",
    "claude-haiku-4.5",
    "claude-sonnet-4",
    "claude-sonnet-4.6",
    "claude-opus-4.6",
    #"gemini-2.5-flash",
    #"gemini-2.5-pro",
    #"gemini-3-flash",
    #"gemini-3.1-pro",
]

# Experiment types
# M0=classify-then-extract, M1=full-multiagent, M6=combined-prompts ablation
# M7=extract-then-verify, M8=ensemble
EXPERIMENT_TYPES = [
     #M0",
     #"M4",
    #"M7",
    #"M8",
     #"M1",
     "M6",
]

# Sampling settings (use same values as notebook 03 for fair comparison)
SAMPLES_PER_TIER = 0              # 0 = full test set, >0 = subsample per tier
NEG_RATIO = 0.7                   # Proportion of negatives (0.0=positive only, 0.7=CUAD natural)
MIN_CONTRACT_CHARS = 0            # Only contracts with at least this many chars (0 = no minimum)
MAX_CONTRACT_CHARS = 400_000      # Skip contracts longer than this

# Run settings
TEMPERATURE = 0.0                 # Generation temperature
MAX_TOKENS = 4096                 # Max output tokens
CONCURRENCY = 10                 # Max parallel API calls per run
IS_OFFICIAL = True               # False = test run (prefixed "test_")
PARALLEL = True                  # Run configs concurrently (only safe across different providers)

# Path to all experiment results (baselines + multi-agent)
RESULTS_DIR = "../experiments/results"

In [6]:
import sys, os, json
from pathlib import Path

sys.path.insert(0, "..")

from dotenv import load_dotenv
load_dotenv("../.env")

from src.experiments.pipeline import ExperimentConfig, run_batch, load_and_select_samples

# Build config matrix: EXPERIMENT_TYPES × MODELS
configs = [
    ExperimentConfig(
        model_key=model,
        run_type=etype,
        samples_per_tier=SAMPLES_PER_TIER,
        neg_ratio=NEG_RATIO,
        min_contract_chars=MIN_CONTRACT_CHARS,
        max_contract_chars=MAX_CONTRACT_CHARS,
        temperature=TEMPERATURE,
        max_tokens=MAX_TOKENS,
        concurrency=CONCURRENCY,
        is_official=IS_OFFICIAL,
    )
    for etype in EXPERIMENT_TYPES
    for model in MODELS
]

print(f"{len(configs)} experiment(s) to run:")
for i, c in enumerate(configs, 1):
    print(f"  {i}. {c.run_type} / {c.model_key} (concurrency={c.effective_concurrency})")

4 experiment(s) to run:
  1. M6 / claude-haiku-4.5 (concurrency=10)
  2. M6 / claude-sonnet-4 (concurrency=10)
  3. M6 / claude-sonnet-4.6 (concurrency=10)
  4. M6 / claude-opus-4.6 (concurrency=10)


## 1. Load and Sample CUAD Data

Identical stratified sampling as notebook 03 (`random.seed(42)`, same tier counts)
to ensure fair comparison against baselines. Loaded once and shared across all runs.

In [7]:
samples = load_and_select_samples(
    samples_per_tier=SAMPLES_PER_TIER,
    neg_ratio=NEG_RATIO,
    min_contract_chars=MIN_CONTRACT_CHARS,
    max_contract_chars=MAX_CONTRACT_CHARS,
)

# Print breakdown
from collections import Counter
tier_counts = Counter(s.tier for s in samples)
pos_counts = Counter(s.tier for s in samples if s.has_clause)
neg_counts = Counter(s.tier for s in samples if not s.has_clause)

total_pos = sum(pos_counts.values())
total_neg = sum(neg_counts.values())
total = total_pos + total_neg

if total == 0:
    print("No samples selected — check MIN_CONTRACT_CHARS / MAX_CONTRACT_CHARS filters.")
else:
    print(f"Selected {total} samples: {total_pos} positive ({total_pos/total:.0%}), {total_neg} negative ({total_neg/total:.0%})\n")
    for tier in ["common", "moderate", "rare"]:
        tp, tn = pos_counts[tier], neg_counts[tier]
        tt = tp + tn
        if tt:
            print(f"  {tier:10s}: {tt:4d} ({tp} pos / {tn} neg — {tn/tt:.0%} neg)")
        else:
            print(f"  {tier:10s}:    0")

    # Contract length distribution
    contract_lengths = {}
    for s in samples:
        if s.contract_title not in contract_lengths:
            contract_lengths[s.contract_title] = len(s.contract_text)

    lengths = sorted(contract_lengths.values())
    n = len(lengths)
    thresholds = [10_000, 25_000, 50_000, 75_000, 100_000, 150_000, 200_000, 300_000]

    print(f"\nContract length distribution ({n} unique contracts):")
    print(f"  Min: {min(lengths):>10,} chars")
    print(f"  Max: {max(lengths):>10,} chars")
    print(f"  Median: {lengths[n//2]:>8,} chars")
    print()
    for t in thresholds:
        count = sum(1 for l in lengths if l > t)
        print(f"  >{t//1000:>3d}k chars: {count:>4d} contracts ({count/n*100:5.1f}%)")

Selected 2999 samples: 1244 positive (41%), 1755 negative (59%)

  common    :  612 (528 pos / 84 neg — 14% neg)
  moderate  : 1680 (504 pos / 1176 neg — 70% neg)
  rare      :  707 (212 pos / 495 neg — 70% neg)

Contract length distribution (102 unique contracts):
  Min:        645 chars
  Max:    300,768 chars
  Median:   25,922 chars

  > 10k chars:   86 contracts ( 84.3%)
  > 25k chars:   55 contracts ( 53.9%)
  > 50k chars:   30 contracts ( 29.4%)
  > 75k chars:   18 contracts ( 17.6%)
  >100k chars:   14 contracts ( 13.7%)
  >150k chars:    5 contracts (  4.9%)
  >200k chars:    4 contracts (  3.9%)
  >300k chars:    1 contracts (  1.0%)


## 2. Run Experiments

Each config runs the full pipeline: setup agents → extract → evaluate → save.

- **M0**: Two-stage classify-then-extract pipeline.
- **M1**: Orchestrator routes to specialist agent via LangGraph.
- **M6**: Single agent with combined specialist prompts (ablation).
- **M7**: B4 extracts → triage → specialist verifies/refines/recovers.
- **M8**: Conservative (B4) + aggressive (specialist) in parallel → resolver adjudicates.
- **Crash-safe**: results append to JSONL after each sample. Re-running resumes.
- **Batch summary**: printed at the end with F1/F2/Jaccard/cost across all runs.

In [8]:
batch_results = await run_batch(
    configs,
    samples=samples,
    output_dir=RESULTS_DIR,
    parallel=PARALLEL,
)


Batch: 4 experiments
  1. M6 / claude-haiku-4.5
  2. M6 / claude-sonnet-4
  3. M6 / claude-sonnet-4.6
  4. M6 / claude-opus-4.6


  M6 | claude-haiku-4.5 | 2999 samples
  run_id: combined_prompts_claude-haiku-4.5_20260416_190207

Pending: 2999 samples (0 already done)



M6 claude-haiku-4.5:   0%|          | 0/2999 [00:00<?, ?it/s]


  M6 | claude-sonnet-4 | 2999 samples
  run_id: combined_prompts_claude-sonnet-4_20260416_190207

Pending: 2999 samples (0 already done)



M6 claude-sonnet-4:   0%|          | 0/2999 [00:00<?, ?it/s]


  M6 | claude-sonnet-4.6 | 2999 samples
  run_id: combined_prompts_claude-sonnet-4.6_20260416_190207

Pending: 2999 samples (0 already done)



M6 claude-sonnet-4.6:   0%|          | 0/2999 [00:00<?, ?it/s]


  M6 | claude-opus-4.6 | 2999 samples
  run_id: combined_prompts_claude-opus-4.6_20260416_190207

Pending: 2999 samples (0 already done)



M6 claude-opus-4.6:   0%|          | 0/2999 [00:00<?, ?it/s]


Completed: 2999 success, 0 failed (0 resumed)
Intermediate saved to: ../experiments/results/combined_prompts_claude-haiku-4.5_20260416_190207_intermediate.jsonl
Total wall time: 1816.1s
  M6 combined_prompts — claude-haiku-4.5
  Samples:       2999
  TP: 1088  FP: 554  FN: 156  TN: 1201

  Precision:     0.663
  Recall:        0.875
  F1:            0.754
  F2:            0.822
  Avg Jaccard:   0.449
  Containment:   0.946
  Span Coverage: 0.812
  Laziness rate: 3.1% (38/1244)

  ContractEval reference (GPT-4.1):
  F1=0.641  F2=0.678  Jaccard=0.472  Laziness=7.1%

  Per-Tier Breakdown
  Tier         TP   FP   FN   TN      F1      F2  Jaccard  Contain  SpanCov
  ------------------------------------------------------------------------------
  common      501   54   27   30   0.925   0.939    0.333    0.984    0.937
  moderate    435  362   69  814   0.669   0.773    0.553    0.955    0.757
  rare        152  138   60  357   0.606   0.668    0.489    0.831    0.630
Model Diagnostics (cla

## 3. Recovery — Generate summary from interrupted run

If a run was cancelled before the final save step, use the cell below to
rebuild the summary and diagnostics from the intermediate JSONL file.

In [13]:
# ============================================================
# RECOVERY — Paste the intermediate JSONL path(s) here
# ============================================================
RECOVER_JSONL_PATHS = [
    # "../experiments/results/extract_then_verify_claude-sonnet-4_20260415_..._intermediate.jsonl",
    "../experiments/results/classify_then_extract_claude-sonnet-4_20260415_232938_intermediate.jsonl",
]

import json
from pathlib import Path
from src.experiments.results import compute_aggregate_metrics, compute_per_tier_metrics, print_metrics, print_diagnostics, save_experiment
from src.models.config import get_model_config, estimate_cost
from src.models.diagnostics import ModelDiagnostics, TokenUsage

for jsonl_path in RECOVER_JSONL_PATHS:
    jsonl_path = Path(jsonl_path)
    if not jsonl_path.exists():
        print(f"SKIP: {jsonl_path} not found")
        continue

    # Load all successful records
    records = []
    with open(jsonl_path) as f:
        for line in f:
            if line.strip():
                rec = json.loads(line)
                if rec.get("status") != "error":
                    records.append(rec)

    if not records:
        print(f"SKIP: {jsonl_path} has no successful records")
        continue

    # Extract run metadata from first record
    first = records[0]
    run_id = first["run_id"]
    model_key = first["model_key"]
    model_id = first.get("model_id", model_key)
    run_type = first.get("baseline_type") or first.get("experiment_type", "B1")
    run_label = first.get("baseline_label") or first.get("experiment_label", "zero_shot")
    run_type_key = "baseline" if "baseline_type" in first else "experiment"
    is_official = first.get("run_mode") == "official"

    print(f"\n{'='*60}")
    print(f"  Recovering: {run_id}")
    print(f"  Records: {len(records)}")
    print(f"{'='*60}\n")

    # Compute metrics
    metrics = compute_aggregate_metrics(records)
    per_tier = compute_per_tier_metrics(records)
    print_metrics(metrics, per_tier, run_type=run_type, run_label=run_label, model_key=model_key)

    # Rebuild diagnostics from per-record usage
    diagnostics = ModelDiagnostics(experiment_id=run_id)
    for rec in records:
        usage = rec.get("usage", {})
        elapsed = rec.get("elapsed_seconds", 0)
        inp = usage.get("input_tokens", 0)
        out = usage.get("output_tokens", 0)
        diagnostics.create_call(
            model_key=model_key,
            model_id=model_id,
            usage=TokenUsage(
                input_tokens=inp,
                output_tokens=out,
                cache_read_tokens=usage.get("cache_read_tokens", 0),
                cache_creation_tokens=usage.get("cache_creation_tokens", 0),
            ),
            latency_ms=elapsed * 1000,
            cost_usd=estimate_cost(model_key, inp, out),
            category=rec.get("category", ""),
        )

    diag_summary = print_diagnostics(diagnostics, model_key)

    # Build prompt/architecture info
    model_cfg = get_model_config(model_key)
    prompt_info = None
    architecture = None

    if run_type_key == "baseline":
        prompt_info = {
            "system_prompt": first.get("input", {}).get("system_prompt"),
            "template_name": run_label,
        }
    else:
        architecture = first.get("architecture")

    # Save summary + diagnostics
    summary_path, diag_path = save_experiment(
        run_id=run_id,
        results=records,
        metrics=metrics,
        per_tier=per_tier,
        diag_summary=diag_summary,
        diagnostics=diagnostics,
        model_key=model_key,
        config=model_cfg,
        run_type=run_type,
        run_label=run_label,
        intermediate_path=jsonl_path,
        temperature=TEMPERATURE,
        max_tokens=MAX_TOKENS,
        samples_per_tier=SAMPLES_PER_TIER,
        max_contract_chars=MAX_CONTRACT_CHARS,
        min_contract_chars=MIN_CONTRACT_CHARS,
        neg_ratio=NEG_RATIO,
        prompt=prompt_info,
        architecture=architecture,
        is_official=is_official,
    )

    print(f"\n  Summary:     {summary_path}")
    print(f"  Diagnostics: {diag_path}")


  Recovering: classify_then_extract_claude-sonnet-4_20260415_232938
  Records: 554

  M0 classify_then_extract — claude-sonnet-4
  Samples:       554
  TP: 447  FP: 32  FN: 29  TN: 46

  Precision:     0.933
  Recall:        0.939
  F1:            0.936
  F2:            0.938
  Avg Jaccard:   0.358
  Containment:   0.948
  Span Coverage: 0.931
  Laziness rate: 4.2% (20/476)

  ContractEval reference (GPT-4.1):
  F1=0.641  F2=0.678  Jaccard=0.472  Laziness=7.1%

  Per-Tier Breakdown
  Tier         TP   FP   FN   TN      F1      F2  Jaccard  Contain  SpanCov
  ------------------------------------------------------------------------------
  common      440   32   28   46   0.936   0.939    0.355    0.950    0.932
  moderate      7    0    1    0   0.933   0.897    0.587    0.875    0.875
  rare          0    0    0    0   0.000   0.000    0.000    0.000    0.000
Model Diagnostics (claude-sonnet-4)
API calls:       554
Success rate:    100%
Input tokens:    312,816,111
Output tokens:   5,

In [ ]:
# ── Load all available results from disk ──
results_dir = Path(RESULTS_DIR)

LABEL_PREFIXES = {
    "B1": "zero_shot",
    "B4": "cot",
    "M0": "classify_then_extract",
    "M1": "multiagent",
    "M6": "combined_prompts",
    "M7": "extract_then_verify",
    "M8": "ensemble",
}

ALL_CONFIGS = list(LABEL_PREFIXES.keys())

def load_latest_run(label_prefix: str) -> tuple[dict | None, list[dict]]:
    """Find the most recent summary + intermediate files for a label prefix."""
    summaries = sorted(results_dir.glob(f"{label_prefix}_*_summary.json"), reverse=True)
    if not summaries:
        return None, []
    summary_path = summaries[0]
    with open(summary_path) as f:
        summary = json.load(f)
    inter_path = summary_path.with_name(
        summary_path.name.replace("_summary.json", "_intermediate.jsonl")
    )
    records = []
    if inter_path.exists():
        with open(inter_path) as f:
            for line in f:
                if line.strip():
                    records.append(json.loads(line))
    return summary, records

all_runs: dict[str, tuple[dict | None, list[dict]]] = {}
for code, prefix in LABEL_PREFIXES.items():
    all_runs[code] = load_latest_run(prefix)

print("Available results:")
for code in ALL_CONFIGS:
    summ, recs = all_runs[code]
    if summ:
        m = summ["metrics"]
        model = summ.get("config", {}).get("model_key", "?")
        print(f"  {code:4s} ({model:20s}): {len(recs):4d} samples | "
              f"F1={m['f1']:.3f}  F2={m['f2']:.3f}  J={m['avg_jaccard']:.3f}")
    else:
        print(f"  {code:4s}: not found")

## 4. Statistical Comparison Against Baselines

Loads **all available results** from the results directory and runs hypothesis
tests on whatever pairs exist. Re-run this section independently — it reads
from disk, not from the batch run above.

**Comparisons run automatically for each multi-agent config (M0–M8) found:**
- **H1**: Mx vs B4 (McNemar + Cohen's d on F2)
- **H2**: Per-tier F2 improvement (rare vs common)
- **H3**: Mx vs M6 (architecture vs prompts)
- **H4**: Trace completeness

In [ ]:
from src.evaluation.statistical import (
    bootstrap_ci, mcnemar_test, wilcoxon_test,
    benjamini_hochberg, cohens_d, format_result,
)


def build_paired_outcomes(records_a: list[dict], records_b: list[dict]):
    """Align two sets of records by sample_id and return paired binary outcomes."""
    by_id_a = {r["sample_id"]: r for r in records_a}
    by_id_b = {r["sample_id"]: r for r in records_b}
    shared_ids = sorted(set(by_id_a) & set(by_id_b))

    correct_a = [by_id_a[sid]["evaluation"]["classification"] in ("TP", "TN") for sid in shared_ids]
    correct_b = [by_id_b[sid]["evaluation"]["classification"] in ("TP", "TN") for sid in shared_ids]
    jaccard_a = [by_id_a[sid]["evaluation"]["jaccard"] for sid in shared_ids]
    jaccard_b = [by_id_b[sid]["evaluation"]["jaccard"] for sid in shared_ids]
    return correct_a, correct_b, jaccard_a, jaccard_b, shared_ids


# ── Identify which multi-agent configs have results ──
MULTIAGENT_CODES = ["M0", "M1", "M7", "M8", "M6"]
available_mx = [code for code in MULTIAGENT_CODES if all_runs[code][1]]
b4_summary, b4_records = all_runs["B4"]
b1_summary, b1_records = all_runs["B1"]
m6_summary, m6_records = all_runs["M6"]

print("=" * 70)
print("  HYPOTHESIS TESTS")
print("=" * 70)

p_values = []

for mx_code in available_mx:
    mx_summary, mx_records = all_runs[mx_code]

    # ── H1: Mx vs B4 ──
    print(f"\n{'─'*60}")
    print(f"  H1: {mx_code} vs B4 (chain-of-thought baseline)")
    print(f"{'─'*60}")
    if mx_records and b4_records:
        correct_mx, correct_b4, jacc_mx, jacc_b4, shared = build_paired_outcomes(mx_records, b4_records)
        print(f"  Paired samples: {len(shared)}")

        if len(shared) >= 5:
            chi2, p_val = mcnemar_test(correct_mx, correct_b4)
            d = cohens_d(jacc_mx, jacc_b4)
            p_values.append(p_val)

            mx_acc = sum(correct_mx) / len(correct_mx)
            b4_acc = sum(correct_b4) / len(correct_b4)
            ci = bootstrap_ci([1 if c else 0 for c in correct_mx])

            print(format_result(f"{mx_code} Accuracy", mx_acc, ci=ci, baseline_value=b4_acc, p_value=p_val, effect_size=d))

            w_stat, w_p = wilcoxon_test(jacc_mx, jacc_b4)
            p_values.append(w_p)
            print(f"  Jaccard Wilcoxon: W={w_stat:.1f}, p={w_p:.4f}")
        else:
            print(f"  Too few paired samples ({len(shared)}) for statistical tests")
    else:
        print(f"  SKIPPED: need both {mx_code} and B4 results")

    # ── H1b: Mx vs B1 ──
    print(f"\n  H1b: {mx_code} vs B1 (zero-shot)")
    if mx_records and b1_records:
        correct_mx, correct_b1, jacc_mx, jacc_b1, shared = build_paired_outcomes(mx_records, b1_records)
        print(f"  Paired samples: {len(shared)}")

        if len(shared) >= 5:
            chi2, p_val = mcnemar_test(correct_mx, correct_b1)
            d = cohens_d(jacc_mx, jacc_b1)
            p_values.append(p_val)

            mx_acc = sum(correct_mx) / len(correct_mx)
            b1_acc = sum(correct_b1) / len(correct_b1)
            print(format_result(f"{mx_code} Accuracy", mx_acc, baseline_value=b1_acc, p_value=p_val, effect_size=d))
        else:
            print(f"  Too few paired samples ({len(shared)})")
    else:
        print(f"  SKIPPED: need both {mx_code} and B1 results")

    # ── H2: Per-tier improvement (Mx vs B4) ──
    print(f"\n  H2: Per-tier improvement ({mx_code} vs B4)")
    if mx_records and b4_records:
        by_id_mx = {r["sample_id"]: r for r in mx_records}
        by_id_b4 = {r["sample_id"]: r for r in b4_records}
        shared_ids = set(by_id_mx) & set(by_id_b4)

        for tier in ["common", "moderate", "rare"]:
            tier_ids = [sid for sid in shared_ids if by_id_mx[sid]["tier"] == tier]
            if not tier_ids:
                print(f"    {tier}: no paired samples")
                continue
            tier_correct_mx = [by_id_mx[sid]["evaluation"]["classification"] in ("TP", "TN") for sid in tier_ids]
            tier_correct_b4 = [by_id_b4[sid]["evaluation"]["classification"] in ("TP", "TN") for sid in tier_ids]
            mx_rate = sum(tier_correct_mx) / len(tier_correct_mx)
            b4_rate = sum(tier_correct_b4) / len(tier_correct_b4)
            delta = mx_rate - b4_rate
            print(f"    {tier:10s}: {mx_code}={mx_rate:.1%}  B4={b4_rate:.1%}  Δ={delta:+.1%}  (n={len(tier_ids)})")
    else:
        print(f"    SKIPPED")

    # ── H3: Mx vs M6 (architecture vs prompts) ──
    if mx_code != "M6":
        print(f"\n  H3: {mx_code} vs M6 (architecture vs prompts)")
        if mx_records and m6_records:
            correct_mx, correct_m6, jacc_mx, jacc_m6, shared = build_paired_outcomes(mx_records, m6_records)
            print(f"  Paired samples: {len(shared)}")

            if len(shared) >= 5:
                chi2, p_val = mcnemar_test(correct_mx, correct_m6)
                d = cohens_d(jacc_mx, jacc_m6)
                p_values.append(p_val)

                mx_acc = sum(correct_mx) / len(correct_mx)
                m6_acc = sum(correct_m6) / len(correct_m6)
                print(format_result(f"{mx_code} Accuracy", mx_acc, baseline_value=m6_acc, p_value=p_val, effect_size=d))

                if p_val < 0.05 and mx_acc > m6_acc:
                    print(f"  => Architecture provides genuine benefit beyond prompting")
                elif p_val >= 0.05:
                    print(f"  => No significant difference")
                else:
                    print(f"  => M6 outperforms {mx_code}")
            else:
                print(f"  Too few paired samples ({len(shared)})")
        else:
            missing = []
            if not mx_records: missing.append(mx_code)
            if not m6_records: missing.append("M6")
            print(f"  SKIPPED (missing: {', '.join(missing)})")

    # ── H4: Trace completeness ──
    print(f"\n  H4: Trace completeness ({mx_code})")
    if mx_records:
        with_trace = [r for r in mx_records if r.get("trace", {}).get("nodes_visited")]
        completeness = len(with_trace) / len(mx_records) if mx_records else 0
        print(f"    Records with trace: {len(with_trace)} / {len(mx_records)}")
        print(f"    Trace completeness: {completeness:.1%} (target: > 90%)")
        print(f"    {'PASS' if completeness > 0.9 else 'FAIL'}")
    else:
        print(f"    SKIPPED")

# ── M7 vs M8 head-to-head ──
m7_summary, m7_records = all_runs["M7"]
m8_summary, m8_records = all_runs["M8"]
if m7_records and m8_records:
    print(f"\n{'─'*60}")
    print(f"  HEAD-TO-HEAD: M7 vs M8")
    print(f"{'─'*60}")
    correct_m7, correct_m8, jacc_m7, jacc_m8, shared = build_paired_outcomes(m7_records, m8_records)
    print(f"  Paired samples: {len(shared)}")
    if len(shared) >= 5:
        chi2, p_val = mcnemar_test(correct_m7, correct_m8)
        d = cohens_d(jacc_m7, jacc_m8)
        p_values.append(p_val)
        m7_acc = sum(correct_m7) / len(correct_m7)
        m8_acc = sum(correct_m8) / len(correct_m8)
        print(format_result("M7 Accuracy", m7_acc, baseline_value=m8_acc, p_value=p_val, effect_size=d))

# ── Multiple comparison correction ──
if p_values:
    print(f"\n{'─'*60}")
    print(f"  Benjamini-Hochberg Correction ({len(p_values)} tests)")
    print(f"{'─'*60}")
    significant = benjamini_hochberg(p_values, alpha=0.05)
    for i, (p, sig) in enumerate(zip(p_values, significant)):
        print(f"  Test {i+1}: p={p:.4f} {'*' if sig else 'ns'}")
    print(f"  Significant after BH correction: {sum(significant)} / {len(significant)}")

## 5. Cross-Configuration Comparison

Summary table across all available runs (loaded from disk).

In [ ]:
# Build comparison table from all loaded summaries
rows = []
for code in ALL_CONFIGS:
    summ, recs = all_runs[code]
    if summ:
        m = summ["metrics"]
        model = summ.get("config", {}).get("model_key", "?")
        rows.append({
            "config": code,
            "model": model,
            "n": sum(m[k] for k in ["tp", "fp", "fn", "tn"]),
            "precision": m["precision"],
            "recall": m["recall"],
            "f1": m["f1"],
            "f2": m["f2"],
            "jaccard": m["avg_jaccard"],
            "laziness": m["laziness_rate"],
        })

if rows:
    print(f"{'='*100}")
    print(f"  Cross-Configuration Comparison")
    print(f"{'='*100}")
    print(f"  {'Config':<6} {'Model':<22} {'N':>5} {'Prec':>7} {'Rec':>7} "
          f"{'F1':>7} {'F2':>7} {'Jaccard':>8} {'Lazy':>7}")
    print(f"  {'-'*93}")
    for r in rows:
        print(f"  {r['config']:<6} {r['model']:<22} {r['n']:>5} "
              f"{r['precision']:>7.3f} {r['recall']:>7.3f} "
              f"{r['f1']:>7.3f} {r['f2']:>7.3f} "
              f"{r['jaccard']:>8.3f} {r['laziness']:>6.1%}")
else:
    print("No results found in", results_dir)

## Next Steps

**Run M7 and M8** (new architectures):
```python
EXPERIMENT_TYPES = ["M7", "M8"]
```

**Run all multi-agent configs** for complete comparison:
```python
EXPERIMENT_TYPES = ["M0", "M1", "M6", "M7", "M8"]
```

**Workflow for complete comparison:**
1. Run notebook 03 with B1 and B4 to establish baselines
2. Run this notebook with M7 + M8 (new architectures)
3. Add M0, M1, M6 for full comparison matrix
4. Re-run sections 4-5 (stats + table) — they auto-load all results from disk

**Architecture descriptions:**
- **M7 (Extract-then-Verify)**: B4 extracts → triage decides → specialist verifies/refines/recovers → grounding. ~1.5 calls/sample.
- **M8 (Ensemble)**: B4 + specialist extract in parallel → merge detects agreement → resolver adjudicates disagreements. ~2.3 calls/sample.

**Output files** (per run):
- `experiments/results/{run_id}_intermediate.jsonl` — full per-sample records
- `experiments/results/{run_id}_summary.json` — config + metrics + compact results
- `experiments/diagnostics/{run_id}_diagnostics.json` — raw API call log